In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

project_id = 'thesis-506605'
dataset_id = 'tables'

client = bigquery.Client(project=project_id)

query = f"""
    SELECT *
    FROM `{project_id}.{dataset_id}.station_features`
    ORDER BY datetime
"""
df_fe = client.query(query).to_dataframe()
df_fe = df_fe.set_index('datetime')  # restore your original index shape

# ready to feed into your model

In [ ]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.preprocessing import MinMaxScaler

# df_fe = pd.read_parquet("df_fe.parquet")

CALENDAR_COLS = ["hour", "month", "day_of_week", "is_weekend", "is_holiday", "is_rainy_season"]
station_cols  = [c for c in df_fe.columns if c not in CALENDAR_COLS]

print(f"Loaded: {df_fe.shape} | {df_fe.index[0].date()} → {df_fe.index[-1].date()}")
print(f"Stations : {station_cols}")
print(f"NaN total: {df_fe[station_cols].isna().sum().sum():,}")

Loaded: (49152, 14) | 2021-01-16 → 2026-08-25
Stations : ['Bendung Katulampa', 'Pos Angke Hulu', 'Pos Cipinang Hulu', 'Pos Depok', 'Pos Krukut Hulu', 'Pos Pesanggrahan', 'Pos Sunter Hulu', 'Waduk Pluit']
NaN total: 74


In [ ]:
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15

n = len(df_fe)
train_end_idx = int(n * TRAIN_RATIO)
val_end_idx   = int(n * (TRAIN_RATIO + VAL_RATIO))

df_train_raw = df_fe.iloc[:train_end_idx]
df_val_raw   = df_fe.iloc[train_end_idx:val_end_idx]
df_test_raw  = df_fe.iloc[val_end_idx:]

print(f"Train : {df_train_raw.index[0].date()} → {df_train_raw.index[-1].date()} ({len(df_train_raw):,} rows, {len(df_train_raw)/n:.1%})")
print(f"Val   : {df_val_raw.index[0].date()} → {df_val_raw.index[-1].date()} ({len(df_val_raw):,} rows, {len(df_val_raw)/n:.1%})")
print(f"Test  : {df_test_raw.index[0].date()} → {df_test_raw.index[-1].date()} ({len(df_test_raw):,} rows, {len(df_test_raw)/n:.1%})")

Train : 2021-01-16 → 2024-12-19 (34,406 rows, 70.0%)
Val   : 2024-12-19 → 2025-10-22 (7,373 rows, 15.0%)
Test  : 2025-10-22 → 2026-08-25 (7,373 rows, 15.0%)


In [ ]:
# Hitung hourly median per stasiun dari training set (raw cm)
hourly_medians = {}

for station in station_cols:
    hourly_medians[station] = (
        df_train_raw[station]
        .groupby(df_train_raw.index.hour)
        .median()
    )

print("Hourly medians per stasiun (cm):")
for station, med in hourly_medians.items():
    print(f"  {station}: min={med.min():.1f} cm, max={med.max():.1f} cm")

Hourly medians per stasiun (cm):
  Bendung Katulampa: min=10.0 cm, max=13.3 cm
  Pos Angke Hulu: min=40.0 cm, max=40.0 cm
  Pos Cipinang Hulu: min=80.0 cm, max=85.0 cm
  Pos Depok: min=90.0 cm, max=95.0 cm
  Pos Krukut Hulu: min=40.0 cm, max=40.0 cm
  Pos Pesanggrahan: min=70.0 cm, max=70.0 cm
  Pos Sunter Hulu: min=90.0 cm, max=90.0 cm
  Waduk Pluit: min=-190.0 cm, max=-180.0 cm


In [ ]:
def fill_nan_hourly_median(df, hourly_medians, station_cols):
    df = df.copy()
    for station in station_cols:
        nan_mask = df[station].isna()
        if nan_mask.sum() == 0:
            continue
        df.loc[nan_mask, station] = df.loc[nan_mask].index.hour.map(
            hourly_medians[station]
        )
        print(f"  [{station}] filled {nan_mask.sum()} NaN")
    return df

df_train_raw = fill_nan_hourly_median(df_train_raw, hourly_medians, station_cols)
df_val_raw   = fill_nan_hourly_median(df_val_raw,   hourly_medians, station_cols)
df_test_raw  = fill_nan_hourly_median(df_test_raw,  hourly_medians, station_cols)

total_nan = (df_train_raw[station_cols].isna().sum().sum() +
             df_val_raw[station_cols].isna().sum().sum() +
             df_test_raw[station_cols].isna().sum().sum())
print(f"\nTotal NaN tersisa: {total_nan}")

  [Bendung Katulampa] filled 4 NaN
  [Pos Angke Hulu] filled 4 NaN
  [Pos Cipinang Hulu] filled 4 NaN
  [Pos Depok] filled 4 NaN
  [Pos Krukut Hulu] filled 4 NaN
  [Pos Pesanggrahan] filled 4 NaN
  [Pos Sunter Hulu] filled 4 NaN
  [Waduk Pluit] filled 46 NaN

Total NaN tersisa: 0


In [ ]:
scalers = {}

for station in station_cols:
    scaler = MinMaxScaler(feature_range=(0, 1))
    valid_vals = df_train_raw[station].dropna().values.reshape(-1, 1)
    scaler.fit(valid_vals)
    scalers[station] = scaler
    print(f"  [{station}] min={scaler.data_min_[0]:.1f} cm, max={scaler.data_max_[0]:.1f} cm")

  [Bendung Katulampa] min=-170.0 cm, max=300.0 cm
  [Pos Angke Hulu] min=0.0 cm, max=365.0 cm
  [Pos Cipinang Hulu] min=-240.0 cm, max=200.0 cm
  [Pos Depok] min=-190.0 cm, max=1000.0 cm
  [Pos Krukut Hulu] min=-190.0 cm, max=660.0 cm
  [Pos Pesanggrahan] min=6.0 cm, max=600.0 cm
  [Pos Sunter Hulu] min=0.0 cm, max=880.0 cm
  [Waduk Pluit] min=-1801.0 cm, max=200.0 cm


In [ ]:
def apply_scalers(df, scalers, station_cols):
    df = df.copy()
    for station in station_cols:
        mask = df[station].notna()
        df.loc[mask, station] = scalers[station].transform(
            df.loc[mask, station].values.reshape(-1, 1)
        ).flatten()
    return df

df_train = apply_scalers(df_train_raw, scalers, station_cols)
df_val   = apply_scalers(df_val_raw,   scalers, station_cols)
df_test  = apply_scalers(df_test_raw,  scalers, station_cols)

print(f"Train range: {df_train[station_cols].min().min():.4f} – {df_train[station_cols].max().max():.4f}")
print(f"Val   range: {df_val[station_cols].min().min():.4f} – {df_val[station_cols].max().max():.4f}")
print(f"Test  range: {df_test[station_cols].min().min():.4f} – {df_test[station_cols].max().max():.4f}")

Train range: 0.0000 – 1.0000
Val   range: -0.0515 – 1.7534
Test  range: -0.3300 – 1.0959


In [ ]:
import os
import pickle

os.makedirs("artifacts", exist_ok=True)

with open("artifacts/scalers.pkl", "wb") as f:
    pickle.dump(scalers, f)

print("Saved: artifacts/scalers.pkl")

Saved: artifacts/scalers.pkl


In [ ]:
# pip install prophet --quiet
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

HORIZONS = [1, 3, 12]  # jam ke depan

In [ ]:
def make_prophet_df(df, target_station, other_stations):
    """Konversi DataFrame ke format Prophet (ds, y, + regressors)."""
    out = df[[target_station] + other_stations].copy()
    out = out.reset_index().rename(columns={
        "datetime": "ds",
        target_station: "y"
    })
    return out


def build_and_train_prophet(df_train_p, other_stations):
    """Fit Prophet dengan semua stasiun lain sebagai regressor."""
    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=True,
        daily_seasonality=True,
        seasonality_mode="additive",
    )
    for station in other_stations:
        m.add_regressor(station)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        m.fit(df_train_p)
    return m


def predict_horizon(model, df_future_p, horizon):
    """
    Prediksi h jam ke depan menggunakan nilai aktual sebagai regressor.
    Karena Prophet tidak native multi-step, kita shift target h langkah
    dan predict satu titik sekaligus (direct forecasting strategy).
    """
    df_shifted = df_future_p.copy()
    df_shifted["ds"] = df_shifted["ds"] + pd.Timedelta(hours=horizon)
    forecast = model.predict(df_shifted)
    return forecast[["ds", "yhat"]].set_index("ds")["yhat"]


def compute_metrics(y_true, y_pred):
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)

    # sMAPE — symmetric, aman untuk nilai mendekati 0
    smape = np.mean(
        2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred) + 1e-8)
    ) * 100

    # NSE — Nash-Sutcliffe Efficiency
    nse = 1 - (np.sum((y_true - y_pred) ** 2) /
               np.sum((y_true - np.mean(y_true)) ** 2 + 1e-8))

    return {"RMSE": rmse, "MAE": mae, "sMAPE": smape, "NSE": nse}

In [ ]:
results = {}
models  = {}

for target in station_cols:
    other_stations = [s for s in station_cols if s != target]
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    df_train_p = make_prophet_df(df_train, target, other_stations)
    df_val_p   = make_prophet_df(df_val,   target, other_stations)

    model = build_and_train_prophet(df_train_p, other_stations)
    models[target] = model

    results[target] = {}

    for h in HORIZONS:
        preds = predict_horizon(model, df_val_p, horizon=h)

        y_true_index = df_val.index + pd.Timedelta(hours=h)
        y_true = df_val[target].reindex(y_true_index).values
        y_pred = preds.reindex(y_true_index).values

        metrics = compute_metrics(y_true, y_pred)
        results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}h → RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")


Target: Bendung Katulampa
  h+ 1h → RMSE=0.0251 | MAE=0.0160 | sMAPE=3.89% | NSE=0.2669
  h+ 3h → RMSE=0.0261 | MAE=0.0164 | sMAPE=3.96% | NSE=0.2088
  h+12h → RMSE=0.0285 | MAE=0.0180 | sMAPE=4.35% | NSE=0.0512

Target: Pos Angke Hulu
  h+ 1h → RMSE=0.1330 | MAE=0.0787 | sMAPE=32.45% | NSE=0.3038
  h+ 3h → RMSE=0.1309 | MAE=0.0772 | sMAPE=31.87% | NSE=0.3266
  h+12h → RMSE=0.1417 | MAE=0.0807 | sMAPE=33.05% | NSE=0.2112

Target: Pos Cipinang Hulu
  h+ 1h → RMSE=0.0278 | MAE=0.0169 | sMAPE=2.28% | NSE=0.1613
  h+ 3h → RMSE=0.0305 | MAE=0.0180 | sMAPE=2.42% | NSE=-0.0055
  h+12h → RMSE=0.0362 | MAE=0.0213 | sMAPE=2.86% | NSE=-0.4137

Target: Pos Depok
  h+ 1h → RMSE=0.0187 | MAE=0.0141 | sMAPE=5.89% | NSE=0.1559
  h+ 3h → RMSE=0.0189 | MAE=0.0141 | sMAPE=5.87% | NSE=0.1371
  h+12h → RMSE=0.0224 | MAE=0.0150 | sMAPE=6.21% | NSE=-0.2148

Target: Pos Krukut Hulu
  h+ 1h → RMSE=0.0217 | MAE=0.0167 | sMAPE=5.63% | NSE=0.4639
  h+ 3h → RMSE=0.0235 | MAE=0.0177 | sMAPE=5.91% | NSE=0.3723
  h+

In [ ]:
# Simpan val results
import json

os.makedirs("artifacts/prophet", exist_ok=True)

with open("artifacts/prophet/val_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved: artifacts/prophet/val_results.json")

Saved: artifacts/prophet/val_results.json


In [ ]:
# Simpan Prophet model objects (agar konsisten dgn LSTM/GRU/TFT yg simpan weights)
import pickle

for station, model in models.items():
    fname = f"{station.replace(' ', '_').lower()}.pkl"
    with open(f"artifacts/prophet/{fname}", "wb") as f:
        pickle.dump(model, f)

print(f"Saved: {len(models)} Prophet models to artifacts/prophet/")

## **LSTM/GRU**

---

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LOOKBACK   = 72
HORIZONS   = [1, 3, 12]
BATCH_SIZE = 64
EPOCHS     = 50
LR         = 1e-3
PATIENCE   = 7       # early stopping

feature_cols = station_cols + CALENDAR_COLS
N_FEATURES   = len(feature_cols)

print(f"Device     : {DEVICE}")
print(f"Features   : {N_FEATURES} ({feature_cols})")
print(f"Lookback   : {LOOKBACK} jam")
print(f"Models     : {len(station_cols)} stasiun × {len(HORIZONS)} horizon = {len(station_cols)*len(HORIZONS)}")

Device     : cuda
Features   : 14 (['Bendung Katulampa', 'Pos Angke Hulu', 'Pos Cipinang Hulu', 'Pos Depok', 'Pos Krukut Hulu', 'Pos Pesanggrahan', 'Pos Sunter Hulu', 'Waduk Pluit', 'hour', 'month', 'day_of_week', 'is_weekend', 'is_holiday', 'is_rainy_season'])
Lookback   : 72 jam
Models     : 8 stasiun × 3 horizon = 24


In [ ]:
class TMADataset(Dataset):
    def __init__(self, df, target_station, feature_cols, lookback, horizon):
        self.X, self.y = [], []
        data   = df[feature_cols].values.astype(np.float32)
        target = df[target_station].values.astype(np.float32)

        for i in range(lookback, len(data) - horizon + 1):
            self.X.append(data[i - lookback:i])          # (lookback, n_features)
            self.y.append(target[i + horizon - 1])        # scalar t+h

        self.X = torch.tensor(np.array(self.X), dtype=torch.float32)
        self.y = torch.tensor(np.array(self.y), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [ ]:
class LSTMForecaster(nn.Module):
    def __init__(self, n_features, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze(-1)   # ambil timestep terakhir

In [ ]:
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    for X, y in loader:
        X, y = X.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        pred = model(X)
        loss = criterion(pred, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(y)
    return total_loss / len(loader.dataset)


def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(DEVICE), y.to(DEVICE)
            pred = model(X)
            total_loss += criterion(pred, y).item() * len(y)
    return total_loss / len(loader.dataset)


def get_predictions(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            preds.append(model(X.to(DEVICE)).cpu().numpy())
            trues.append(y.numpy())
    return np.concatenate(preds), np.concatenate(trues)

In [ ]:
lstm_results = {}   # {station: {horizon: metrics}}
lstm_models  = {}   # {station: {horizon: model}}

criterion = nn.MSELoss()

for target in station_cols:
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    lstm_models[target] = {}
    lstm_results[target] = {}

    for h in HORIZONS:
        # Dataset & loader
        train_ds = TMADataset(df_train, target, feature_cols, LOOKBACK, h)
        val_ds   = TMADataset(df_val,   target, feature_cols, LOOKBACK, h)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        # Model
        model = LSTMForecaster(n_features=N_FEATURES).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        # Early stopping
        best_val_loss = float("inf")
        best_state    = None
        patience_ctr  = 0

        for epoch in range(EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_loss   = eval_epoch(model, val_loader, criterion)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr  = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"  h+{h:>2}h | early stop epoch {epoch+1:>3} | "
                          f"val_loss={best_val_loss:.6f}")
                    break

        # Load best weights
        model.load_state_dict(best_state)
        lstm_models[target][h] = model

        # Metrics pada val set
        y_pred, y_true = get_predictions(model, val_loader)
        metrics = compute_metrics(y_true, y_pred)
        lstm_results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}h → RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")


Target: Bendung Katulampa
  h+ 1h | early stop epoch  30 | val_loss=0.000207
  h+ 1h → RMSE=0.0144 | MAE=0.0064 | sMAPE=1.53% | NSE=0.7571
  h+ 3h | early stop epoch  30 | val_loss=0.000407
  h+ 3h → RMSE=0.0202 | MAE=0.0104 | sMAPE=2.49% | NSE=0.5235
  h+12h | early stop epoch  25 | val_loss=0.000641
  h+12h → RMSE=0.0253 | MAE=0.0158 | sMAPE=3.77% | NSE=0.2511

Target: Pos Angke Hulu
  h+ 1h | early stop epoch  17 | val_loss=0.001287
  h+ 1h → RMSE=0.0359 | MAE=0.0142 | sMAPE=5.90% | NSE=0.9497
  h+ 3h | early stop epoch  19 | val_loss=0.004112
  h+ 3h → RMSE=0.0641 | MAE=0.0298 | sMAPE=12.75% | NSE=0.8393
  h+12h | early stop epoch  11 | val_loss=0.015916
  h+12h → RMSE=0.1262 | MAE=0.0729 | sMAPE=29.83% | NSE=0.3788

Target: Pos Cipinang Hulu
  h+ 1h | early stop epoch  35 | val_loss=0.000253
  h+ 1h → RMSE=0.0159 | MAE=0.0061 | sMAPE=0.83% | NSE=0.7293
  h+ 3h | early stop epoch  33 | val_loss=0.000502
  h+ 3h → RMSE=0.0224 | MAE=0.0098 | sMAPE=1.31% | NSE=0.4617
  h+12h | early 

In [ ]:
import json
import os
import numpy as np
import torch

os.makedirs("artifacts/lstm", exist_ok=True)

def convert_numpy_floats(obj):
    # """Recursively converts numpy floats to Python floats in a dictionary."""
    if isinstance(obj, dict):
        return {k: convert_numpy_floats(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy_floats(elem) for elem in obj]
    elif isinstance(obj, (np.float32, np.float64, np.float16)):
        return float(obj)
    return obj

# Convert numpy floats in lstm_results before dumping
serializable_lstm_results = convert_numpy_floats(lstm_results)

with open("artifacts/lstm/val_results.json", "w") as f:
    json.dump(serializable_lstm_results, f, indent=2)

# Simpan model weights
for station, horizons in lstm_models.items():
    for h, model in horizons.items():
        fname = f"{station.replace(' ', '_').lower()}_h{h}.pt"
        torch.save(model.state_dict(), f"artifacts/lstm/{fname}")

print(f"Saved: {len(station_cols) * len(HORIZONS)} models + val_results.json")

Saved: 24 models + val_results.json


## **GRU**

In [ ]:
class GRUForecaster(nn.Module):
    def __init__(self, n_features, hidden_size=128, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :]).squeeze(-1)

In [ ]:
gru_results = {}
gru_models  = {}

criterion = nn.MSELoss()

for target in station_cols:
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    gru_models[target] = {}
    gru_results[target] = {}

    for h in HORIZONS:
        train_ds = TMADataset(df_train, target, feature_cols, LOOKBACK, h)
        val_ds   = TMADataset(df_val,   target, feature_cols, LOOKBACK, h)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = GRUForecaster(n_features=N_FEATURES).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_val_loss = float("inf")
        best_state    = None
        patience_ctr  = 0

        for epoch in range(EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_loss   = eval_epoch(model, val_loader, criterion)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr  = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"  h+{h:>2}h | early stop epoch {epoch+1:>3} | "
                          f"val_loss={best_val_loss:.6f}")
                    break

        model.load_state_dict(best_state)
        gru_models[target][h] = model

        y_pred, y_true = get_predictions(model, val_loader)
        metrics = compute_metrics(y_true, y_pred)
        gru_results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}h → RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")


Target: Bendung Katulampa
  h+ 1h | early stop epoch  23 | val_loss=0.000210
  h+ 1h → RMSE=0.0145 | MAE=0.0070 | sMAPE=1.67% | NSE=0.7546
  h+ 3h | early stop epoch  29 | val_loss=0.000415
  h+ 3h → RMSE=0.0204 | MAE=0.0106 | sMAPE=2.53% | NSE=0.5139
  h+12h | early stop epoch  25 | val_loss=0.000632
  h+12h → RMSE=0.0251 | MAE=0.0156 | sMAPE=3.73% | NSE=0.2616

Target: Pos Angke Hulu
  h+ 1h | early stop epoch  13 | val_loss=0.001344
  h+ 1h → RMSE=0.0367 | MAE=0.0136 | sMAPE=5.75% | NSE=0.9475
  h+ 3h | early stop epoch  12 | val_loss=0.004281
  h+ 3h → RMSE=0.0654 | MAE=0.0317 | sMAPE=14.21% | NSE=0.8328
  h+12h | early stop epoch  13 | val_loss=0.016334
  h+12h → RMSE=0.1278 | MAE=0.0754 | sMAPE=30.16% | NSE=0.3624

Target: Pos Cipinang Hulu
  h+ 1h | early stop epoch  32 | val_loss=0.000257
  h+ 1h → RMSE=0.0160 | MAE=0.0064 | sMAPE=0.87% | NSE=0.7246
  h+ 3h | early stop epoch  26 | val_loss=0.000509
  h+ 3h → RMSE=0.0226 | MAE=0.0101 | sMAPE=1.36% | NSE=0.4537
  h+12h | early 

In [ ]:
os.makedirs("artifacts/gru", exist_ok=True)

# Convert numpy floats in gru_results before dumping
serializable_gru_results = convert_numpy_floats(gru_results)

with open("artifacts/gru/val_results.json", "w") as f:
    json.dump(serializable_gru_results, f, indent=2)

for station, horizons in gru_models.items():
    for h, model in horizons.items():
        fname = f"{station.replace(' ', '_').lower()}_h{h}.pt"
        torch.save(model.state_dict(), f"artifacts/gru/{fname}")

print(f"Saved: {len(station_cols) * len(HORIZONS)} GRU models + val_results.json")

Saved: 24 GRU models + val_results.json


## Model Comparison — Validation Set

Tiga model dievaluasi sebagai kandidat: **Prophet** (baseline), **LSTM**, dan **GRU**
across 8 stasiun dan 3 horizon prediksi (h+1, h+3, h+12 jam).

---

### NSE (↑ higher is better)

| Station | Model | h+1 | h+3 | h+12 |
|---|---|---|---|---|
| Bendung Katulampa | Prophet | 0.1251 | 0.0638 | -0.1177 |
| | LSTM | 0.8132 | 0.5870 | 0.3238 |
| | GRU | 0.8085 | 0.5594 | 0.3230 |
| Pos Angke Hulu | Prophet | 0.3557 | 0.3786 | 0.2637 |
| | LSTM | 0.9683 | 0.8901 | 0.4576 |
| | GRU | 0.9689 | 0.8863 | 0.4435 |
| Pos Cipinang Hulu | Prophet | -0.1287 | -0.2690 | -0.8143 |
| | LSTM | 0.6878 | 0.4501 | -0.1060 |
| | GRU | 0.6496 | 0.4433 | -0.1844 |
| Pos Depok | Prophet | 0.6045 | 0.5847 | 0.2647 |
| | LSTM | 0.8710 | 0.7659 | 0.3841 |
| | GRU | 0.8661 | 0.7586 | 0.3533 |
| Pos Krukut Hulu | Prophet | 0.6444 | 0.5643 | 0.0050 |
| | LSTM | 0.8791 | 0.7076 | 0.1283 |
| | GRU | 0.8961 | 0.7138 | 0.1795 |
| Pos Pesanggrahan | Prophet | -0.5218 | -0.6088 | -1.0741 |
| | LSTM | 0.8787 | 0.7511 | 0.2512 |
| | GRU | 0.8619 | 0.7428 | 0.2536 |
| Pos Sunter Hulu | Prophet | 0.1911 | 0.1408 | -0.1315 |
| | LSTM | 0.8930 | 0.6954 | 0.3033 |
| | GRU | 0.8892 | 0.7039 | 0.2982 |
| Waduk Pluit | Prophet | -0.1652 | -0.1652 | -0.1786 |
| | LSTM | 0.0364 | 0.0437 | 0.0162 |
| | GRU | 0.0043 | 0.0071 | 0.0111 |

---

### RMSE (↓ lower is better)

| Station | Model | h+1 | h+3 | h+12 |
|---|---|---|---|---|
| Bendung Katulampa | Prophet | 0.0271 | 0.0280 | 0.0306 |
| | LSTM | 0.0126 | 0.0187 | 0.0239 |
| | GRU | 0.0127 | 0.0193 | 0.0239 |
| Pos Angke Hulu | Prophet | 0.1150 | 0.1130 | 0.1231 |
| | LSTM | 0.0256 | 0.0477 | 0.1060 |
| | GRU | 0.0254 | 0.0485 | 0.1073 |
| Pos Cipinang Hulu | Prophet | 0.0298 | 0.0316 | 0.0378 |
| | LSTM | 0.0158 | 0.0209 | 0.0297 |
| | GRU | 0.0167 | 0.0210 | 0.0307 |
| Pos Depok | Prophet | 0.0136 | 0.0140 | 0.0186 |
| | LSTM | 0.0078 | 0.0105 | 0.0170 |
| | GRU | 0.0079 | 0.0106 | 0.0174 |
| Pos Krukut Hulu | Prophet | 0.0176 | 0.0195 | 0.0295 |
| | LSTM | 0.0103 | 0.0161 | 0.0277 |
| | GRU | 0.0096 | 0.0159 | 0.0269 |
| Pos Pesanggrahan | Prophet | 0.1252 | 0.1287 | 0.1462 |
| | LSTM | 0.0354 | 0.0507 | 0.0880 |
| | GRU | 0.0378 | 0.0515 | 0.0878 |
| Pos Sunter Hulu | Prophet | 0.0302 | 0.0311 | 0.0357 |
| | LSTM | 0.0110 | 0.0185 | 0.0280 |
| | GRU | 0.0112 | 0.0183 | 0.0281 |
| Waduk Pluit | Prophet | 0.0143 | 0.0143 | 0.0144 |
| | LSTM | 0.0131 | 0.0131 | 0.0133 |
| | GRU | 0.0133 | 0.0133 | 0.0133 |

---

### sMAPE % (↓ lower is better)

| Station | Model | h+1 | h+3 | h+12 |
|---|---|---|---|---|
| Bendung Katulampa | Prophet | 5.06 | 5.22 | 5.64 |
| | LSTM | 1.45 | 2.37 | 3.40 |
| | GRU | 1.36 | 2.52 | 3.55 |
| Pos Angke Hulu | Prophet | 33.07 | 32.68 | 34.34 |
| | LSTM | 6.01 | 10.75 | 32.23 |
| | GRU | 5.09 | 11.55 | 28.47 |
| Pos Cipinang Hulu | Prophet | 2.57 | 2.70 | 3.18 |
| | LSTM | 0.78 | 1.32 | 2.49 |
| | GRU | 1.02 | 1.34 | 2.84 |
| Pos Depok | Prophet | 3.53 | 3.56 | 4.34 |
| | LSTM | 1.57 | 2.35 | 4.14 |
| | GRU | 1.60 | 2.36 | 4.05 |
| Pos Krukut Hulu | Prophet | 3.62 | 3.86 | 5.52 |
| | LSTM | 1.74 | 2.96 | 4.52 |
| | GRU | 1.48 | 2.74 | 5.02 |
| Pos Pesanggrahan | Prophet | 47.05 | 46.96 | 47.78 |
| | LSTM | 3.21 | 6.85 | 15.07 |
| | GRU | 4.78 | 7.37 | 16.03 |
| Pos Sunter Hulu | Prophet | 21.13 | 21.40 | 23.21 |
| | LSTM | 3.72 | 7.83 | 13.47 |
| | GRU | 4.38 | 7.10 | 14.02 |
| Waduk Pluit | Prophet | 0.81 | 0.81 | 0.81 |
| | LSTM | 0.60 | 0.64 | 0.57 |
| | GRU | 0.59 | 0.55 | 0.55 |

---

### Key Findings

- **LSTM dan GRU jauh mengungguli Prophet** di hampir semua stasiun dan horizon.
  Prophet gagal capture dinamika spiky TMA — NSE mayoritas negatif atau mendekati 0.
- **LSTM vs GRU hampir identik** — selisih NSE <0.05 di sebagian besar kasus,
  tidak ada yang konsisten menang di semua stasiun.
- **Degradasi performa konsisten di h+12** across semua model — inheren pada
  prediksi TMA jangka menengah, bukan artefak model.
- **Waduk Pluit** adalah outlier — NSE mendekati 0 di semua model bukan karena
  model jelek, melainkan variansi data sangat rendah (std ≈ 0.013 scaled).
  RMSE dan sMAPE justru terkecil di antara semua stasiun.
- **Prophet dipertahankan sebagai baseline** — hasil inferior-nya memotivasi
  penggunaan deep learning untuk forecasting TMA.

### Next Step
Evaluasi dilanjutkan dengan **TFT (Temporal Fusion Transformer)** di val set,
kemudian perbandingan final LSTM vs GRU vs TFT di test set.

## **TFT (Temporal Fusion Transformer)**

Implementasi TFT mengikuti arsitektur Lim et al. (2021) dan disusun agar
*kompatibel penuh* dengan pipeline LSTM/GRU di atas — memakai `TMADataset`,
`train_epoch`/`eval_epoch`/`get_predictions`, `compute_metrics`, serta konvensi
per-stasiun × per-horizon yang sama, sehingga hasilnya langsung dapat
dibandingkan.

Komponen utama:

- **Variable Selection Network (VSN)** — memberi bobot per-fitur (TMA antar
  stasiun + fitur kalender) di setiap timestep, sehingga model dapat menekan
  fitur yang tidak relevan alih-alih memperlakukan semua input setara seperti
  LSTM/GRU biasa. Ini juga menjadi sumber *interpretability* (importance fitur).
- **Gated Residual Network (GRN)** — blok non-linear dengan gating (GLU) +
  koneksi residual + LayerNorm; memungkinkan informasi melewati transformasi
  hanya bila diperlukan, menstabilkan pelatihan pada sinyal TMA yang *spiky*.
- **LSTM encoder** — memproses seluruh jendela *lookback* (72 jam) untuk
  memberi ringkasan konteks lokal (local processing) sebelum atensi.
- **Interpretable Multi-Head Attention** — menangkap dependensi jangka panjang
  antar timestep di dalam lookback; menjadi mekanisme yang diharapkan
  memperbaiki kelemahan degradasi h+12 pada LSTM/GRU.
- **Static enrichment** — pada setup ini kita tidak memiliki kovariat statis
  eksternal, sehingga enrichment memakai konteks agregat dari encoder
  (tetap disertakan agar arsitektur lengkap dan mudah diperluas).

Karena tugasnya adalah *direct single-step forecasting* (prediksi skalar
`t+h`), head TFT mereduksi keluaran atensi pada posisi terakhir menjadi satu
nilai — identik dengan cara LSTM/GRU mengambil `out[:, -1, :]`.


In [ ]:
# --- TFT building blocks -----------------------------------------------------
import torch
import torch.nn as nn
import torch.nn.functional as F


class GatedLinearUnit(nn.Module):
    """GLU: mengatur seberapa besar informasi diteruskan (gating)."""
    def __init__(self, input_size, hidden_size=None, dropout=0.1):
        super().__init__()
        hidden_size = hidden_size or input_size
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(input_size, hidden_size * 2)
        self.hidden_size = hidden_size

    def forward(self, x):
        x = self.dropout(x)
        x = self.fc(x)
        return F.glu(x, dim=-1)   # split terakhir jadi (value, gate)


class GatedResidualNetwork(nn.Module):
    """
    GRN(x) = LayerNorm(x + GLU(ELU(W1 x + b1) -> W2))
    Blok inti TFT: transformasi non-linear yang bisa "dilewati" via gating.
    """
    def __init__(self, input_size, hidden_size, output_size=None, dropout=0.1,
                 context_size=None):
        super().__init__()
        output_size = output_size or input_size
        self.input_size = input_size
        self.output_size = output_size

        self.fc1 = nn.Linear(input_size, hidden_size)
        self.context = (nn.Linear(context_size, hidden_size, bias=False)
                        if context_size is not None else None)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.glu = GatedLinearUnit(hidden_size, output_size, dropout)
        self.layer_norm = nn.LayerNorm(output_size)

        # proyeksi skip bila dimensi input != output
        self.skip = (nn.Linear(input_size, output_size)
                     if input_size != output_size else None)

    def forward(self, x, context=None):
        residual = x if self.skip is None else self.skip(x)
        h = self.fc1(x)
        if self.context is not None and context is not None:
            h = h + self.context(context)
        h = F.elu(h)
        h = self.fc2(h)
        h = self.glu(h)
        return self.layer_norm(h + residual)


class VariableSelectionNetwork(nn.Module):
    """
    Memberi bobot softmax per-fitur di tiap timestep, lalu menggabungkan
    representasi tiap fitur (masing-masing lewat GRN) dengan bobot tsb.
    Input : (batch, time, n_features)
    Output: (batch, time, hidden_size), plus bobot (batch, time, n_features)
    """
    def __init__(self, n_features, hidden_size, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.hidden_size = hidden_size

        # GRN untuk menghitung bobot seleksi (dari seluruh fitur flatten)
        self.flattened_grn = GatedResidualNetwork(
            n_features, hidden_size, n_features, dropout
        )
        # GRN per-fitur (skalar -> hidden)
        self.per_feature_grn = nn.ModuleList([
            GatedResidualNetwork(1, hidden_size, hidden_size, dropout)
            for _ in range(n_features)
        ])

    def forward(self, x):
        # x: (B, T, F)
        weights = self.flattened_grn(x)                 # (B, T, F)
        weights = torch.softmax(weights, dim=-1).unsqueeze(-1)  # (B, T, F, 1)

        transformed = torch.stack([
            self.per_feature_grn[i](x[..., i:i + 1])    # (B, T, hidden)
            for i in range(self.n_features)
        ], dim=-2)                                       # (B, T, F, hidden)

        combined = (transformed * weights).sum(dim=-2)   # (B, T, hidden)
        return combined, weights.squeeze(-1)


class InterpretableMultiHeadAttention(nn.Module):
    """
    Multi-head attention dgn value bersama antar head (versi interpretable TFT).
    """
    def __init__(self, hidden_size, n_heads, dropout=0.1):
        super().__init__()
        assert hidden_size % n_heads == 0
        self.n_heads = n_heads
        self.d_head = hidden_size // n_heads

        self.q = nn.Linear(hidden_size, hidden_size)
        self.k = nn.Linear(hidden_size, hidden_size)
        self.v = nn.Linear(hidden_size, self.d_head)     # value dibagi antar head
        self.out = nn.Linear(self.d_head, hidden_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v(x).unsqueeze(1)                        # (B,1,T,d_head)

        scores = (q @ k.transpose(-2, -1)) / (self.d_head ** 0.5)
        attn = torch.softmax(scores, dim=-1)
        attn = self.dropout(attn)

        ctx = attn @ v                                    # (B,H,T,d_head)
        ctx = ctx.mean(dim=1)                             # rata-rata antar head
        return self.out(ctx), attn


class TFTForecaster(nn.Module):
    """
    Temporal Fusion Transformer untuk direct single-step forecasting.
    Signature forward identik dgn LSTM/GRUForecaster: x (B, LOOKBACK, F) -> (B,)
    """
    def __init__(self, n_features, hidden_size=64, n_heads=4,
                 dropout=0.2, lstm_layers=1):
        super().__init__()
        self.vsn = VariableSelectionNetwork(n_features, hidden_size, dropout)

        self.lstm_encoder = nn.LSTM(
            input_size=hidden_size, hidden_size=hidden_size,
            num_layers=lstm_layers, batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        # gate + norm setelah encoder (post-LSTM gating TFT)
        self.post_lstm_glu = GatedLinearUnit(hidden_size, hidden_size, dropout)
        self.post_lstm_norm = nn.LayerNorm(hidden_size)

        # static enrichment memakai konteks agregat encoder
        self.enrichment = GatedResidualNetwork(
            hidden_size, hidden_size, hidden_size, dropout,
            context_size=hidden_size,
        )
        self.attention = InterpretableMultiHeadAttention(
            hidden_size, n_heads, dropout
        )
        self.post_attn_glu = GatedLinearUnit(hidden_size, hidden_size, dropout)
        self.post_attn_norm = nn.LayerNorm(hidden_size)

        self.position_wise = GatedResidualNetwork(
            hidden_size, hidden_size, hidden_size, dropout
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x, return_weights=False):
        # 1) variable selection
        selected, var_weights = self.vsn(x)              # (B,T,H)

        # 2) local processing (LSTM encoder)
        lstm_out, (h_n, _) = self.lstm_encoder(selected)
        lstm_out = self.post_lstm_norm(
            self.post_lstm_glu(lstm_out) + selected      # residual skip
        )

        # 3) static enrichment (konteks = hidden terakhir encoder)
        context = h_n[-1].unsqueeze(1).expand(-1, lstm_out.size(1), -1)
        enriched = self.enrichment(lstm_out, context)    # (B,T,H)

        # 4) temporal self-attention
        attn_out, attn_weights = self.attention(enriched)
        attn_out = self.post_attn_norm(
            self.post_attn_glu(attn_out) + enriched
        )

        # 5) position-wise feed-forward + head pada timestep terakhir
        out = self.position_wise(attn_out)
        pred = self.head(out[:, -1, :]).squeeze(-1)       # (B,)

        if return_weights:
            return pred, var_weights, attn_weights
        return pred


In [ ]:
# --- Hyperparameter TFT (selaras dgn LSTM/GRU) ------------------------------
TFT_HIDDEN   = 64
TFT_HEADS    = 4
TFT_DROPOUT  = 0.2
TFT_LSTM_LAYERS = 1

print(f"TFT hidden={TFT_HIDDEN} | heads={TFT_HEADS} | "
      f"dropout={TFT_DROPOUT} | lstm_layers={TFT_LSTM_LAYERS}")
print(f"Reuse: LOOKBACK={LOOKBACK}, HORIZONS={HORIZONS}, "
      f"BATCH_SIZE={BATCH_SIZE}, EPOCHS={EPOCHS}, LR={LR}, PATIENCE={PATIENCE}")


TFT hidden=64 | heads=4 | dropout=0.2 | lstm_layers=1
Reuse: LOOKBACK=72, HORIZONS=[1, 3, 12], BATCH_SIZE=64, EPOCHS=50, LR=0.001, PATIENCE=7


In [ ]:
tft_results = {}
tft_models  = {}

criterion = nn.MSELoss()

for target in station_cols:
    print(f"\n{'='*50}")
    print(f"Target: {target}")
    print(f"{'='*50}")

    tft_models[target]  = {}
    tft_results[target] = {}

    for h in HORIZONS:
        train_ds = TMADataset(df_train, target, feature_cols, LOOKBACK, h)
        val_ds   = TMADataset(df_val,   target, feature_cols, LOOKBACK, h)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
        val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)

        model = TFTForecaster(
            n_features=N_FEATURES,
            hidden_size=TFT_HIDDEN,
            n_heads=TFT_HEADS,
            dropout=TFT_DROPOUT,
            lstm_layers=TFT_LSTM_LAYERS,
        ).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        best_val_loss = float("inf")
        best_state    = None
        patience_ctr  = 0

        for epoch in range(EPOCHS):
            train_loss = train_epoch(model, train_loader, optimizer, criterion)
            val_loss   = eval_epoch(model, val_loader, criterion)

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                patience_ctr  = 0
            else:
                patience_ctr += 1
                if patience_ctr >= PATIENCE:
                    print(f"  h+{h:>2}h | early stop epoch {epoch+1:>3} | "
                          f"val_loss={best_val_loss:.6f}")
                    break

        model.load_state_dict(best_state)
        tft_models[target][h] = model

        y_pred, y_true = get_predictions(model, val_loader)
        metrics = compute_metrics(y_true, y_pred)
        tft_results[target][f"h+{h}"] = metrics

        print(f"  h+{h:>2}h → RMSE={metrics['RMSE']:.4f} | "
              f"MAE={metrics['MAE']:.4f} | "
              f"sMAPE={metrics['sMAPE']:.2f}% | "
              f"NSE={metrics['NSE']:.4f}")



Target: Bendung Katulampa
  h+ 1h | early stop epoch  29 | val_loss=0.000194
  h+ 1h → RMSE=0.0139 | MAE=0.0044 | sMAPE=1.03% | NSE=0.7730
  h+ 3h | early stop epoch  26 | val_loss=0.000391
  h+ 3h → RMSE=0.0198 | MAE=0.0089 | sMAPE=2.09% | NSE=0.5424
  h+12h | early stop epoch  26 | val_loss=0.000574
  h+12h → RMSE=0.0240 | MAE=0.0136 | sMAPE=3.24% | NSE=0.3295

Target: Pos Angke Hulu
  h+ 1h | early stop epoch  24 | val_loss=0.001042
  h+ 1h → RMSE=0.0323 | MAE=0.0126 | sMAPE=5.09% | NSE=0.9593
  h+ 3h | early stop epoch  34 | val_loss=0.003728
  h+ 3h → RMSE=0.0611 | MAE=0.0284 | sMAPE=11.35% | NSE=0.8543
  h+12h | early stop epoch  15 | val_loss=0.016212
  h+12h → RMSE=0.1273 | MAE=0.0701 | sMAPE=28.00% | NSE=0.3672

Target: Pos Cipinang Hulu
  h+ 1h | early stop epoch   9 | val_loss=0.000251
  h+ 1h → RMSE=0.0159 | MAE=0.0057 | sMAPE=0.77% | NSE=0.7304
  h+ 3h | early stop epoch  10 | val_loss=0.000502
  h+ 3h → RMSE=0.0224 | MAE=0.0086 | sMAPE=1.15% | NSE=0.4615
  h+12h | early 

In [ ]:
os.makedirs("artifacts/tft", exist_ok=True)

# Convert numpy floats in tft_results before dumping
serializable_tft_results = convert_numpy_floats(tft_results)

with open("artifacts/tft/val_results.json", "w") as f:
    json.dump(serializable_tft_results, f, indent=2)

for station, horizons in tft_models.items():
    for h, model in horizons.items():
        fname = f"{station.replace(' ', '_').lower()}_h{h}.pt"
        torch.save(model.state_dict(), f"artifacts/tft/{fname}")

print(f"Saved: {len(station_cols) * len(HORIZONS)} TFT models + val_results.json")


Saved: 24 TFT models + val_results.json


## Streamlined Pipeline — Unified Model Registry

Bagian ini menyatukan keempat kandidat (**Prophet, LSTM, GRU, TFT**) di bawah
satu antarmuka tunggal agar setiap model menempuh langkah yang identik:
train → validate → simpan → evaluasi. Tujuannya menghilangkan duplikasi kode
antar sel dan menjamin perbandingan yang *apple-to-apple*.

Registry di bawah mendefinisikan, untuk tiap model:

- `builder` — fungsi yang mengembalikan instance model baru (untuk model NN),
- `kind` — `"nn"` (PyTorch, memakai `TMADataset`) atau `"prophet"`,
- direktori artefak dan hasil validasi yang sudah tersimpan.

Fungsi `run_nn_model(...)` menggeneralisasi loop LSTM/GRU/TFT menjadi satu
pemanggilan; blok-blok eksperimen di atas tetap dipertahankan untuk keterbacaan
naratif skripsi, sedangkan runner ini dipakai untuk reproduksi & evaluasi
test-set pada tahap pemilihan model.


In [ ]:
from typing import Callable, Dict

def run_nn_model(model_builder: Callable[[], nn.Module],
                 df_tr, df_va,
                 station_cols, feature_cols,
                 lookback, horizons,
                 batch_size=BATCH_SIZE, epochs=EPOCHS,
                 lr=LR, patience=PATIENCE, verbose=True):
    """
    Loop train+val terpadu untuk model NN apa pun ber-signature (B,T,F)->(B,).
    Mengembalikan (results, models) dengan struktur identik seperti sel LSTM/GRU/TFT.
    """
    results, trained = {}, {}
    criterion = nn.MSELoss()

    for target in station_cols:
        if verbose:
            print(f"\n{'='*50}\nTarget: {target}\n{'='*50}")
        trained[target], results[target] = {}, {}

        for h in horizons:
            train_ds = TMADataset(df_tr, target, feature_cols, lookback, h)
            val_ds   = TMADataset(df_va, target, feature_cols, lookback, h)
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
            val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False)

            model = model_builder().to(DEVICE)
            optimizer = torch.optim.Adam(model.parameters(), lr=lr)

            best_val_loss, best_state, patience_ctr = float("inf"), None, 0
            for epoch in range(epochs):
                train_epoch(model, train_loader, optimizer, criterion)
                val_loss = eval_epoch(model, val_loader, criterion)
                if val_loss < best_val_loss:
                    best_val_loss = val_loss
                    best_state = {k: v.clone() for k, v in model.state_dict().items()}
                    patience_ctr = 0
                else:
                    patience_ctr += 1
                    if patience_ctr >= patience:
                        break

            model.load_state_dict(best_state)
            trained[target][h] = model

            y_pred, y_true = get_predictions(model, val_loader)
            metrics = compute_metrics(y_true, y_pred)
            results[target][f"h+{h}"] = metrics
            if verbose:
                print(f"  h+{h:>2}h → RMSE={metrics['RMSE']:.4f} | "
                      f"MAE={metrics['MAE']:.4f} | sMAPE={metrics['sMAPE']:.2f}% | "
                      f"NSE={metrics['NSE']:.4f}")
    return results, trained


# Registry kandidat model. `builder` dipakai ulang saat evaluasi test-set.
MODEL_REGISTRY: Dict[str, dict] = {
    "prophet": {
        "kind": "prophet",
        "artifact_dir": "artifacts/prophet",
        "val_results": "artifacts/prophet/val_results.json",
    },
    "lstm": {
        "kind": "nn",
        "builder": lambda: LSTMForecaster(n_features=N_FEATURES),
        "artifact_dir": "artifacts/lstm",
        "val_results": "artifacts/lstm/val_results.json",
    },
    "gru": {
        "kind": "nn",
        "builder": lambda: GRUForecaster(n_features=N_FEATURES),
        "artifact_dir": "artifacts/gru",
        "val_results": "artifacts/gru/val_results.json",
    },
    "tft": {
        "kind": "nn",
        "builder": lambda: TFTForecaster(
            n_features=N_FEATURES, hidden_size=TFT_HIDDEN,
            n_heads=TFT_HEADS, dropout=TFT_DROPOUT,
            lstm_layers=TFT_LSTM_LAYERS,
        ),
        "artifact_dir": "artifacts/tft",
        "val_results": "artifacts/tft/val_results.json",
    },
}

print("Registered models:", list(MODEL_REGISTRY.keys()))


Registered models: ['prophet', 'lstm', 'gru', 'tft']


## Model Selection — Aggregate Validation Metrics

Semua hasil validasi dikumpulkan ke satu DataFrame panjang
(`model × station × horizon × metric`). Model terbaik dipilih berdasarkan
metrik agregat yang dapat dikonfigurasi — default **mean NSE** (semakin tinggi
semakin baik) dirata-ratakan di seluruh stasiun dan horizon, karena NSE adalah
metrik standar hidrologi yang menormalkan variansi tiap stasiun sehingga adil
untuk dibandingkan lintas lokasi.


In [ ]:
import pandas as pd

# Kumpulkan hasil val dari memori bila ada, else fallback ke file JSON.
_in_memory = {
    "prophet": globals().get("results"),        # sel Prophet menyimpan ke `results`
    "lstm":    globals().get("lstm_results"),
    "gru":     globals().get("gru_results"),
    "tft":     globals().get("tft_results"),
}

def _load_val_results(model_name):
    res = _in_memory.get(model_name)
    if res:
        return res
    path = MODEL_REGISTRY[model_name]["val_results"]
    if os.path.exists(path):
        with open(path) as f:
            return json.load(f)
    return None

rows = []
for model_name in MODEL_REGISTRY:
    res = _load_val_results(model_name)
    if not res:
        print(f"  [skip] no val results for {model_name}")
        continue
    for station, horizons in res.items():
        for hkey, metrics in horizons.items():
            for metric_name, value in metrics.items():
                rows.append({
                    "model": model_name, "station": station,
                    "horizon": hkey, "metric": metric_name,
                    "value": float(value),
                })

val_long = pd.DataFrame(rows)
print(f"Collected {len(val_long)} metric rows "
      f"across {val_long['model'].nunique()} models.")
val_long.head()


Collected 384 metric rows across 4 models.


,model,station,horizon,metric,value
0,prophet,Bendung Katulampa,h+1,RMSE,0.025127
1,prophet,Bendung Katulampa,h+1,MAE,0.016048
2,prophet,Bendung Katulampa,h+1,sMAPE,3.887017
3,prophet,Bendung Katulampa,h+1,NSE,0.266912
4,prophet,Bendung Katulampa,h+3,RMSE,0.026067


In [ ]:
# --- Konfigurasi pemilihan model -------------------------------------------
SELECTION_METRIC = "NSE"     # opsi: NSE | RMSE | MAE | sMAPE
HIGHER_IS_BETTER = {"NSE": True, "RMSE": False, "MAE": False, "sMAPE": False}

agg = (val_long[val_long["metric"] == SELECTION_METRIC]
       .groupby("model")["value"]
       .mean()
       .sort_values(ascending=not HIGHER_IS_BETTER[SELECTION_METRIC]))

print(f"Mean {SELECTION_METRIC} per model (validation):")
for m, v in agg.items():
    print(f"  {m:<8} {v:>8.4f}")

BEST_MODEL = agg.index[0]
print(f"\n✔ Best model by mean {SELECTION_METRIC}: '{BEST_MODEL}'")

# Ringkasan pivot (model x horizon) untuk metrik terpilih
pivot = (val_long[val_long["metric"] == SELECTION_METRIC]
         .pivot_table(index="model", columns="horizon",
                      values="value", aggfunc="mean"))
pivot


Mean NSE per model (validation):
  tft        0.5510
  lstm       0.5136
  gru        0.5074
  prophet    0.1489

✔ Best model by mean NSE: 'tft'


horizon,h+1,h+12,h+3
model,,,
gru,0.751867,0.172451,0.597961
lstm,0.760647,0.178059,0.601980
prophet,0.296562,-0.088908,0.238979
tft,0.803349,0.212893,0.636815


## Retrain / Evaluate Best Model on Test Set

Model pemenang dievaluasi pada **test set** (periode paling akhir, belum pernah
disentuh) untuk mengukur performa generalisasi sebenarnya. Untuk model NN,
prediksi dihasilkan ulang pada test loader; untuk Prophet, dipakai strategi
*direct forecasting* yang sama seperti pada sel baseline. Prediksi test —
beserta metriknya — inilah yang akan ditulis ke BigQuery.

> Catatan: model NN di-*retrain* pada `df_train` lalu dievaluasi di `df_test`
> memakai bobot validation-best, konsisten dengan cara ketiga model NN dilatih.
> Bila objek model dari sel di atas masih tersedia di memori, bobot tersebut
> digunakan langsung tanpa pelatihan ulang.


In [ ]:
def _trained_store(model_name):
    return {
        "lstm": globals().get("lstm_models"),
        "gru":  globals().get("gru_models"),
        "tft":  globals().get("tft_models"),
    }.get(model_name)


def predict_test_nn(model_name, target, h):
    """Prediksi test-set utk satu (station, horizon). Reuse bobot bila ada."""
    test_ds = TMADataset(df_test, target, feature_cols, LOOKBACK, h)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    store = _trained_store(model_name)
    model = None
    if store and target in store and h in store[target]:
        model = store[target][h]
    else:
        # rebuild + load bobot dari artefak
        model = MODEL_REGISTRY[model_name]["builder"]().to(DEVICE)
        fname = f"{target.replace(' ', '_').lower()}_h{h}.pt"
        path = os.path.join(MODEL_REGISTRY[model_name]["artifact_dir"], fname)
        model.load_state_dict(torch.load(path, map_location=DEVICE))

    # index waktu untuk tiap sampel: pertama valid = LOOKBACK+h-1
    idx = df_test.index[LOOKBACK + h - 1 : len(df_test)]
    y_pred, y_true = get_predictions(model, test_loader)
    return idx[:len(y_pred)], y_true, y_pred


def predict_test_prophet(target, h):
    """Direct forecasting Prophet pada test-set (butuh objek `models` di memori)."""
    prophet_models = globals().get("models")
    if not prophet_models or target not in prophet_models:
        raise RuntimeError("Prophet models tidak tersedia di memori; "
                           "jalankan ulang sel training Prophet.")
    other = [s for s in station_cols if s != target]
    df_test_p = make_prophet_df(df_test, target, other)
    preds = predict_horizon(prophet_models[target], df_test_p, horizon=h)

    y_true_index = df_test.index + pd.Timedelta(hours=h)
    y_true = df_test[target].reindex(y_true_index).values
    y_pred = preds.reindex(y_true_index).values
    mask = ~np.isnan(y_true) & ~np.isnan(y_pred)
    return y_true_index[mask], y_true[mask], y_pred[mask]


# --- Jalankan evaluasi test-set untuk BEST_MODEL ----------------------------
test_results = {}
test_predictions = []   # baris panjang utk BigQuery

for target in station_cols:
    test_results[target] = {}
    for h in HORIZONS:
        if MODEL_REGISTRY[BEST_MODEL]["kind"] == "prophet":
            idx, y_true, y_pred = predict_test_prophet(target, h)
        else:
            idx, y_true, y_pred = predict_test_nn(BEST_MODEL, target, h)

        metrics = compute_metrics(y_true, y_pred)
        test_results[target][f"h+{h}"] = metrics

        for ts, yt, yp in zip(idx, y_true, y_pred):
            test_predictions.append({
                "model": BEST_MODEL,
                "station": target,
                "horizon": int(h),
                "target_datetime": pd.Timestamp(ts),
                "y_true": float(yt),
                "y_pred": float(yp),
            })

    row = " | ".join(
        f"h+{h}: NSE={test_results[target][f'h+{h}']['NSE']:.3f}"
        for h in HORIZONS
    )
    print(f"{target:<20} {row}")

test_pred_df = pd.DataFrame(test_predictions)
print(f"\nTest predictions: {test_pred_df.shape[0]:,} rows "
      f"for best model '{BEST_MODEL}'.")
test_pred_df.head()


Bendung Katulampa    h+1: NSE=0.874 | h+3: NSE=0.729 | h+12: NSE=0.591
Pos Angke Hulu       h+1: NSE=0.978 | h+3: NSE=0.922 | h+12: NSE=0.604
Pos Cipinang Hulu    h+1: NSE=0.724 | h+3: NSE=0.453 | h+12: NSE=-0.212
Pos Depok            h+1: NSE=0.918 | h+3: NSE=0.828 | h+12: NSE=0.606
Pos Krukut Hulu      h+1: NSE=0.862 | h+3: NSE=0.695 | h+12: NSE=0.145
Pos Pesanggrahan     h+1: NSE=0.920 | h+3: NSE=0.786 | h+12: NSE=0.404
Pos Sunter Hulu      h+1: NSE=0.946 | h+3: NSE=0.842 | h+12: NSE=0.517
Waduk Pluit          h+1: NSE=0.337 | h+3: NSE=0.278 | h+12: NSE=0.043

Test predictions: 175,120 rows for best model 'tft'.


,model,station,horizon,target_datetime,y_true,y_pred
0,tft,Bendung Katulampa,1,2025-10-25 19:00:00,0.382979,0.386052
1,tft,Bendung Katulampa,1,2025-10-25 20:00:00,0.382979,0.386041
2,tft,Bendung Katulampa,1,2025-10-25 21:00:00,0.382979,0.386013
3,tft,Bendung Katulampa,1,2025-10-25 22:00:00,0.382979,0.385981
4,tft,Bendung Katulampa,1,2025-10-25 23:00:00,0.382979,0.385925


In [ ]:
# Persist test metrics + predictions lokal (audit trail sebelum ke BigQuery)
os.makedirs(f"artifacts/{BEST_MODEL}", exist_ok=True)

with open(f"artifacts/{BEST_MODEL}/test_results.json", "w") as f:
    json.dump(convert_numpy_floats(test_results), f, indent=2)

test_pred_df.to_parquet(f"artifacts/{BEST_MODEL}/test_predictions.parquet",
                        index=False)

# Metrik test dalam bentuk panjang (utk tabel metrik BigQuery)
metric_rows = []
for station, horizons in test_results.items():
    for hkey, metrics in horizons.items():
        for metric_name, value in metrics.items():
            metric_rows.append({
                "model": BEST_MODEL, "station": station,
                "horizon": int(hkey.replace("h+", "")),
                "metric": metric_name, "value": float(value),
            })
test_metrics_df = pd.DataFrame(metric_rows)

print(f"Saved: artifacts/{BEST_MODEL}/test_results.json "
      f"+ test_predictions.parquet")
print(f"Test metrics rows: {len(test_metrics_df)}")
test_metrics_df.head()


Saved: artifacts/tft/test_results.json + test_predictions.parquet
Test metrics rows: 96


,model,station,horizon,metric,value
0,tft,Bendung Katulampa,1,RMSE,0.011339
1,tft,Bendung Katulampa,1,MAE,0.004183
2,tft,Bendung Katulampa,1,sMAPE,0.946167
3,tft,Bendung Katulampa,1,NSE,0.874307
4,tft,Bendung Katulampa,3,RMSE,0.016654


## Write Best Model Results to BigQuery

Prediksi test-set dan metrik dari model pemenang ditulis ke BigQuery. Isi
konfigurasi `PROJECT_ID` / `DATASET` / tabel sesuai lingkungan Anda. Autentikasi
mengikuti mekanisme standar Google Cloud (Application Default Credentials, mis.
`gcloud auth application-default login`, `GOOGLE_APPLICATION_CREDENTIALS`, atau
kredensial bawaan Vertex AI / Colab).

Dua tabel ditulis:

- `*_predictions` — satu baris per (model, station, horizon, target_datetime)
  berisi `y_true` dan `y_pred`.
- `*_metrics` — satu baris per (model, station, horizon, metric).

Skema diberikan eksplisit dan mode tulis default `WRITE_TRUNCATE` (ganti seluruh
isi tabel). Ubah ke `WRITE_APPEND` bila ingin menumpuk hasil beberapa run.


In [ ]:
# --- Konfigurasi BigQuery ----------------------------------------------------
PROJECT_ID = "thesis-506605"
DATASET    = "tables"
LOCATION   = "US"
PREDICTIONS_TABLE = f"{PROJECT_ID}.{DATASET}.best_model_predictions"
METRICS_TABLE     = f"{PROJECT_ID}.{DATASET}.best_model_metrics"

WRITE_DISPOSITION = "WRITE_TRUNCATE"      # atau "WRITE_APPEND"

print("Predictions ->", PREDICTIONS_TABLE)
print("Metrics     ->", METRICS_TABLE)
print("Best model  :", BEST_MODEL, "| disposition:", WRITE_DISPOSITION)


Predictions -> thesis-506605.tables.best_model_predictions
Metrics     -> thesis-506605.tables.best_model_metrics
Best model  : tft | disposition: WRITE_TRUNCATE


In [ ]:
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery

client = bigquery.Client(project=PROJECT_ID, location=LOCATION)

# Pastikan dataset ada (buat bila belum).
ds_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET}")
ds_ref.location = LOCATION
try:
    client.get_dataset(f"{PROJECT_ID}.{DATASET}")
    print(f"Dataset {DATASET} sudah ada.")
except Exception:
    client.create_dataset(ds_ref, exists_ok=True)
    print(f"Dataset {DATASET} dibuat.")


# --- Schema eksplisit --------------------------------------------------------
pred_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("target_datetime", "TIMESTAMP", mode="REQUIRED"),
    bigquery.SchemaField("y_true", "FLOAT", mode="NULLABLE"),
    bigquery.SchemaField("y_pred", "FLOAT", mode="NULLABLE"),
]
metric_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("metric", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("value", "FLOAT", mode="NULLABLE"),
]


def load_df(df, table_id, schema):
    job_config = bigquery.LoadJobConfig(
        schema=schema,
        write_disposition=WRITE_DISPOSITION,
    )
    job = client.load_table_from_dataframe(df, table_id, job_config=job_config)
    job.result()   # tunggu selesai
    tbl = client.get_table(table_id)
    print(f"  {table_id}: {tbl.num_rows:,} rows")


print("Uploading to BigQuery ...")
# target_datetime -> pastikan tz-naive UTC agar konsisten di BQ
_pred = test_pred_df.copy()
_pred["target_datetime"] = pd.to_datetime(_pred["target_datetime"])

load_df(_pred, PREDICTIONS_TABLE, pred_schema)
load_df(test_metrics_df, METRICS_TABLE, metric_schema)
print("\n✔ BigQuery write selesai.")


## Persist Model Weights to BigQuery

Folder `artifacts/` bersifat sementara (hilang saat runtime Colab disconnect),
jadi bobot model pemenang (`BEST_MODEL`) ditulis juga sebagai baris di BigQuery
agar permanen dan bisa diambil ulang dari mesin/session manapun.

Setiap baris menyimpan satu file bobot (per station, dan per horizon untuk
model NN) sebagai kolom `BYTES`. Bila objek model tidak ada di memori (mis.
notebook baru saja dibuka ulang), fallback membaca file `.pt`/`.pkl` dari
`artifacts/{BEST_MODEL}/`.

In [ ]:
import io
import pickle

WEIGHTS_TABLE = f"{PROJECT_ID}.{DATASET}.model_weights"

weights_schema = [
    bigquery.SchemaField("model", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("station", "STRING", mode="REQUIRED"),
    bigquery.SchemaField("horizon", "INTEGER", mode="NULLABLE"),
    bigquery.SchemaField("weights", "BYTES", mode="REQUIRED"),
    bigquery.SchemaField("n_bytes", "INTEGER", mode="REQUIRED"),
    bigquery.SchemaField("saved_at", "TIMESTAMP", mode="REQUIRED"),
]


def serialize_torch_state(state_dict):
    """Serialize state_dict PyTorch ke bytes (in-memory, tanpa file sementara)."""
    buf = io.BytesIO()
    torch.save(state_dict, buf)
    return buf.getvalue()


def serialize_pickle(obj):
    return pickle.dumps(obj)


# saved_at tz-naive UTC, konsisten dgn target_datetime di tabel predictions
saved_at = pd.Timestamp.now(tz="UTC").tz_localize(None)

weight_rows = []

if MODEL_REGISTRY[BEST_MODEL]["kind"] == "prophet":
    prophet_models = globals().get("models")
    for station in station_cols:
        if prophet_models and station in prophet_models:
            blob = serialize_pickle(prophet_models[station])
        else:
            fname = f"{station.replace(' ', '_').lower()}.pkl"
            path = os.path.join(MODEL_REGISTRY[BEST_MODEL]["artifact_dir"], fname)
            with open(path, "rb") as f:
                blob = f.read()
        weight_rows.append({
            "model": BEST_MODEL, "station": station, "horizon": None,
            "weights": blob, "n_bytes": len(blob), "saved_at": saved_at,
        })
else:
    store = _trained_store(BEST_MODEL)
    for station in station_cols:
        for h in HORIZONS:
            if store and station in store and h in store[station]:
                blob = serialize_torch_state(store[station][h].state_dict())
            else:
                fname = f"{station.replace(' ', '_').lower()}_h{h}.pt"
                path = os.path.join(MODEL_REGISTRY[BEST_MODEL]["artifact_dir"], fname)
                with open(path, "rb") as f:
                    blob = f.read()
            weight_rows.append({
                "model": BEST_MODEL, "station": station, "horizon": int(h),
                "weights": blob, "n_bytes": len(blob), "saved_at": saved_at,
            })

weights_df = pd.DataFrame(weight_rows)
total_mb = weights_df["n_bytes"].sum() / 1e6
print(f"Menyiapkan {len(weights_df)} baris bobot ({total_mb:.2f} MB total) utk model '{BEST_MODEL}'.")

load_df(weights_df, WEIGHTS_TABLE, weights_schema)
print(f"\n\u2714 Bobot model '{BEST_MODEL}' tersimpan permanen di {WEIGHTS_TABLE}.")